<a href="https://colab.research.google.com/github/avrilfiona29/Parallel_Computing/blob/main/NLP_ASSIGNMENT_1_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1 — Install/import libraries

In [ ]:
!pip install nltk scikit-learn numpy

Cell 2 — Download WordNet

In [ ]:
import nltk

nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

Cell 3 — Imports

In [ ]:
import numpy as np
from nltk.corpus import wordnet as wn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

CELL 4 — Create our document collection

In [ ]:
documents = [
    "The automobile industry is growing rapidly.",
    "I bought a new car yesterday.",
    "The vehicle requires regular maintenance.",
    "Python is a popular programming language.",
    "The motorcar was invented many years ago.",
    "The computer processes data efficiently.",
    "Automobiles are becoming more environmentally friendly.",
    "The software developer created a new application."
]

doc_ids = [f"D{i+1}" for i in range(len(documents))]

print("Number of documents:", len(documents))

for doc_id, document in zip(doc_ids, documents):
    print(doc_id, ":", document)

Number of documents: 8
D1 : The automobile industry is growing rapidly.
D2 : I bought a new car yesterday.
D3 : The vehicle requires regular maintenance.
D4 : Python is a popular programming language.
D5 : The motorcar was invented many years ago.
D6 : The computer processes data efficiently.
D7 : Automobiles are becoming more environmentally friendly.
D8 : The software developer created a new application.


CELL 5 — Test WordNet

In [ ]:
from nltk.corpus import wordnet as wn

word = "car"

synsets = wn.synsets(word)

print("Word:", word)
print("\nWordNet Synsets:")

for synset in synsets:
    print(synset.name(), "->", synset.definition())

Word: car

WordNet Synsets:
car.n.01 -> a motor vehicle with four wheels; usually propelled by an internal combustion engine
car.n.02 -> a wheeled vehicle adapted to the rails of railroad
car.n.03 -> the compartment that is suspended from an airship and that carries personnel and the cargo and the power plant
car.n.04 -> where passengers ride up and down
cable_car.n.01 -> a conveyance for passengers or freight on a cable railway


CELL 6 — Get synonyms

In [ ]:
word = "car"

synonyms = set()

for synset in wn.synsets(word):
    for lemma in synset.lemmas():
        synonyms.add(lemma.name().replace("_", " "))

print("Synonyms / related terms for:", word)
print(sorted(synonyms))

Synonyms / related terms for: car
['auto', 'automobile', 'cable car', 'car', 'elevator car', 'gondola', 'machine', 'motorcar', 'railcar', 'railroad car', 'railway car']


CELL 7 — Smart WordNet Query Expansion

In [ ]:
def expand_query(query):
    expanded_terms = set()

    words = query.lower().split()

    for word in words:
        expanded_terms.add(word)

        synsets = wn.synsets(word)

        if synsets:
            # Use the first/main meaning of the word
            main_synset = synsets[0]

            for lemma in main_synset.lemmas():
                expanded_terms.add(
                    lemma.name().replace("_", " ")
                )

    return list(expanded_terms)

In [ ]:
query = "car"

expanded_terms = expand_query(query)

print("Original Query:", query)
print("Expanded Terms:", expanded_terms)

Original Query: car
Expanded Terms: ['car', 'machine', 'motorcar', 'auto', 'automobile']


CELL 8 — Create TF-IDF vectors

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer()

# Convert all documents into TF-IDF vectors
document_vectors = vectorizer.fit_transform(documents)

print("TF-IDF matrix shape:", document_vectors.shape)

TF-IDF matrix shape: (8, 38)


CELL 9 — Convert our expanded query into TF-IDF

In [ ]:
expanded_query = " ".join(expanded_terms)

query_vector = vectorizer.transform([expanded_query])

print("Original Query:", query)
print("Expanded Query:", expanded_query)

Original Query: car
Expanded Query: car machine motorcar auto automobile


CELL 10 — Calculate similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    query_vector,
    document_vectors
).flatten()

print("Similarity Scores:")

for doc_id, score in zip(doc_ids, similarities):
    print(doc_id, ":", round(score, 4))

Similarity Scores:
D1 : 0.2578
D2 : 0.3001
D3 : 0.0
D4 : 0.0
D5 : 0.2297
D6 : 0.0
D7 : 0.0
D8 : 0.0


CELL 11 — Rank the documents

In [ ]:
ranked_indices = np.argsort(similarities)[::-1]

print("Ranked Search Results:\n")

for rank, index in enumerate(ranked_indices, start=1):
    print(
        f"Rank {rank}: {doc_ids[index]} "
        f"(Score: {similarities[index]:.4f})"
    )
    print("       ", documents[index])

Ranked Search Results:

Rank 1: D2 (Score: 0.3001)
        I bought a new car yesterday.
Rank 2: D1 (Score: 0.2578)
        The automobile industry is growing rapidly.
Rank 3: D5 (Score: 0.2297)
        The motorcar was invented many years ago.
Rank 4: D8 (Score: 0.0000)
        The software developer created a new application.
Rank 5: D6 (Score: 0.0000)
        The computer processes data efficiently.
Rank 6: D7 (Score: 0.0000)
        Automobiles are becoming more environmentally friendly.
Rank 7: D3 (Score: 0.0000)
        The vehicle requires regular maintenance.
Rank 8: D4 (Score: 0.0000)
        Python is a popular programming language.


CELL 12 — Improve WordNet expansion

In [ ]:
def expand_query_semantically(query):
    expanded_terms = set()

    words = query.lower().split()

    for word in words:
        expanded_terms.add(word)

        synsets = wn.synsets(word)

        if synsets:
            # Main meaning of the word
            main_synset = synsets[0]

            # Add synonyms
            for lemma in main_synset.lemmas():
                expanded_terms.add(
                    lemma.name().replace("_", " ")
                )

            # Add hypernyms
            for hypernym in main_synset.hypernyms():
                for lemma in hypernym.lemmas():
                    expanded_terms.add(
                        lemma.name().replace("_", " ")
                    )

    return sorted(expanded_terms)

In [ ]:
expanded_terms = expand_query_semantically("car")

print("Expanded Query Terms:")
print(expanded_terms)

Expanded Query Terms:
['auto', 'automobile', 'automotive vehicle', 'car', 'machine', 'motor vehicle', 'motorcar']


CELL 13 — Recalculate TF-IDF

In [ ]:
expanded_query = " ".join(expanded_terms)

query_vector = vectorizer.transform([expanded_query])

similarities = cosine_similarity(
    query_vector,
    document_vectors
).flatten()

print("Original Query:", query)
print("Expanded Query:", expanded_query)

print("\nNew Similarity Scores:")

for doc_id, score in zip(doc_ids, similarities):
    print(doc_id, ":", round(score, 4))

Original Query: car
Expanded Query: auto automobile automotive vehicle car machine motor vehicle motorcar

New Similarity Scores:
D1 : 0.1687
D2 : 0.1964
D3 : 0.3639
D4 : 0.0
D5 : 0.1504
D6 : 0.0
D7 : 0.0
D8 : 0.0


CELL 14 — Add lemmatization

In [ ]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    words = word_tokenize(text.lower())

    processed_words = []

    for word in words:
        if word.isalpha():
            processed_words.append(
                lemmatizer.lemmatize(word)
            )

    return " ".join(processed_words)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


CELL 15 — Preprocess all documents

In [ ]:
processed_documents = [
    preprocess_text(document)
    for document in documents
]

print("Processed Documents:\n")

for doc_id, document in zip(doc_ids, processed_documents):
    print(doc_id, ":", document)

Processed Documents:

D1 : the automobile industry is growing rapidly
D2 : i bought a new car yesterday
D3 : the vehicle requires regular maintenance
D4 : python is a popular programming language
D5 : the motorcar wa invented many year ago
D6 : the computer process data efficiently
D7 : automobile are becoming more environmentally friendly
D8 : the software developer created a new application


CELL 16 — Rebuild TF-IDF

In [ ]:
vectorizer = TfidfVectorizer()

document_vectors = vectorizer.fit_transform(
    processed_documents
)

processed_query = preprocess_text(query)

expanded_terms = expand_query_semantically(processed_query)

expanded_query = " ".join(
    preprocess_text(term)
    for term in expanded_terms
)

query_vector = vectorizer.transform(
    [expanded_query]
)

similarities = cosine_similarity(
    query_vector,
    document_vectors
).flatten()

print("Original Query:", query)
print("Expanded Query:", expanded_query)

Original Query: car
Expanded Query: auto automobile automotive vehicle car machine motor vehicle motorcar


CELL 17 — Final ranking

In [ ]:
ranked_indices = np.argsort(similarities)[::-1]

print("\nFinal Ranked Search Results:\n")

for rank, index in enumerate(ranked_indices, start=1):
    print(
        f"Rank {rank}: {doc_ids[index]} "
        f"(Score: {similarities[index]:.4f})"
    )
    print("       ", documents[index])


Final Ranked Search Results:

Rank 1: D3 (Score: 0.3719)
        The vehicle requires regular maintenance.
Rank 2: D2 (Score: 0.2007)
        I bought a new car yesterday.
Rank 3: D5 (Score: 0.1537)
        The motorcar was invented many years ago.
Rank 4: D1 (Score: 0.1249)
        The automobile industry is growing rapidly.
Rank 5: D7 (Score: 0.1136)
        Automobiles are becoming more environmentally friendly.
Rank 6: D8 (Score: 0.0000)
        The software developer created a new application.
Rank 7: D6 (Score: 0.0000)
        The computer processes data efficiently.
Rank 8: D4 (Score: 0.0000)
        Python is a popular programming language.


CELL 18 — Define the relevant documents

In [ ]:
relevant_documents = {"D1", "D2", "D3", "D5", "D7"}

print("Relevant Documents:", relevant_documents)

Relevant Documents: {'D1', 'D3', 'D5', 'D2', 'D7'}


CELL 19 — Decide which documents are retrieved

In [ ]:
retrieved_indices = [
    index for index in ranked_indices
    if similarities[index] > 0
]

retrieved_documents = [
    doc_ids[index]
    for index in retrieved_indices
]

print("Retrieved Documents:", retrieved_documents)

Retrieved Documents: ['D3', 'D2', 'D5', 'D1', 'D7']


CELL 20 — Precision

In [ ]:
retrieved_set = set(retrieved_documents)

true_positives = len(
    retrieved_set & relevant_documents
)

precision = true_positives / len(retrieved_set)

print("True Positives:", true_positives)
print("Precision:", round(precision, 4))

True Positives: 5
Precision: 1.0


CELL 21 — Recall

In [ ]:
recall = true_positives / len(relevant_documents)

print("Recall:", round(recall, 4))

Recall: 1.0


CELL 22 — F1 Score

In [ ]:
if precision + recall == 0:
    f1 = 0
else:
    f1 = 2 * (precision * recall) / (precision + recall)

print("F1 Score:", round(f1, 4))

F1 Score: 1.0


CELL 23 — Average Precision

In [ ]:
def average_precision(ranked_documents, relevant_documents):

    relevant_count = 0
    precision_sum = 0

    for rank, doc_id in enumerate(ranked_documents, start=1):

        if doc_id in relevant_documents:

            relevant_count += 1

            precision_at_rank = relevant_count / rank

            precision_sum += precision_at_rank

    if len(relevant_documents) == 0:
        return 0

    return precision_sum / len(relevant_documents)

In [ ]:
ranked_documents = [
    doc_ids[index]
    for index in ranked_indices
]

ap = average_precision(
    ranked_documents,
    relevant_documents
)

print("Average Precision:", round(ap, 4))

Average Precision: 1.0


CELL 24 — MAP

In [ ]:
map_score = ap

print("MAP:", round(map_score, 4))

MAP: 1.0


PART 2 — Multiple Queries

We're going to use the same document collection, but define three queries and their ground-truth relevant documents.

**Query 1: car**

D1 → automobile

D2 → car

D3 → vehicle

D5 → motorcar

D7 → automobiles

**Query 2: computer**

D6 → computer

**Query 3: programming**

D4 → programming

D8 → software

CELL 25 — Define multiple queries

In [ ]:
queries = {
    "car": {"D1", "D2", "D3", "D5", "D7"},
    "computer": {"D6"},
    "programming": {"D4", "D8"}
}

print("Queries and Relevant Documents:\n")

for query, relevant in queries.items():
    print(query, "->", relevant)

Queries and Relevant Documents:

car -> {'D1', 'D3', 'D5', 'D2', 'D7'}
computer -> {'D6'}
programming -> {'D4', 'D8'}


CELL 26 — Create a function for retrieval

In [ ]:
def retrieve_documents(query):

    # Preprocess query
    processed_query = preprocess_text(query)

    # Expand query using WordNet
    expanded_terms = expand_query_semantically(processed_query)

    # Preprocess expanded terms
    expanded_query = " ".join(
        preprocess_text(term)
        for term in expanded_terms
    )

    # Convert expanded query to TF-IDF
    query_vector = vectorizer.transform(
        [expanded_query]
    )

    # Calculate cosine similarity
    similarities = cosine_similarity(
        query_vector,
        document_vectors
    ).flatten()

    # Rank documents
    ranked_indices = np.argsort(similarities)[::-1]

    ranked_documents = [
        doc_ids[index]
        for index in ranked_indices
    ]

    return ranked_documents, similarities, expanded_query

CELL 27 — Test the function

Let's test it with car first.

In [ ]:
ranked_documents, similarities, expanded_query = retrieve_documents("car")

print("Query:", "car")
print("Expanded Query:", expanded_query)

print("\nRanked Results:")

for rank, doc_id in enumerate(ranked_documents, start=1):

    index = doc_ids.index(doc_id)

    print(
        f"{rank}. {doc_id} "
        f"(Score: {similarities[index]:.4f})"
    )

Query: car
Expanded Query: auto automobile automotive vehicle car machine motor vehicle motorcar

Ranked Results:
1. D3 (Score: 0.3719)
2. D2 (Score: 0.2007)
3. D5 (Score: 0.1537)
4. D1 (Score: 0.1249)
5. D7 (Score: 0.1136)
6. D8 (Score: 0.0000)
7. D6 (Score: 0.0000)
8. D4 (Score: 0.0000)


CELL 28 — Function for Precision, Recall and F1

In [ ]:
def calculate_metrics(
    ranked_documents,
    similarities,
    relevant_documents
):

    retrieved_documents = []

    for doc_id in ranked_documents:

        index = doc_ids.index(doc_id)

        if similarities[index] > 0:
            retrieved_documents.append(doc_id)

    retrieved_set = set(retrieved_documents)
    relevant_set = set(relevant_documents)

    true_positives = len(
        retrieved_set & relevant_set
    )

    precision = (
        true_positives / len(retrieved_set)
        if len(retrieved_set) > 0 else 0
    )

    recall = (
        true_positives / len(relevant_set)
        if len(relevant_set) > 0 else 0
    )

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = (
            2 * precision * recall
        ) / (precision + recall)

    return precision, recall, f1

CELL 29 — Calculate AP

In [ ]:
def calculate_ap(
    ranked_documents,
    relevant_documents
):

    relevant_count = 0
    precision_sum = 0

    for rank, doc_id in enumerate(
        ranked_documents,
        start=1
    ):

        if doc_id in relevant_documents:

            relevant_count += 1

            precision_at_rank = (
                relevant_count / rank
            )

            precision_sum += precision_at_rank

    if len(relevant_documents) == 0:
        return 0

    return precision_sum / len(relevant_documents)

CELL 30 — Evaluate all queries

In [ ]:
results = []

for query, relevant_documents in queries.items():

    ranked_documents, similarities, expanded_query = retrieve_documents(query)

    precision, recall, f1 = calculate_metrics(
        ranked_documents,
        similarities,
        relevant_documents
    )

    ap = calculate_ap(
        ranked_documents,
        relevant_documents
    )

    results.append({
        "Query": query,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "AP": ap
    })

    print("\nQuery:", query)
    print("Expanded Query:", expanded_query)
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1:", round(f1, 4))
    print("AP:", round(ap, 4))


Query: car
Expanded Query: auto automobile automotive vehicle car machine motor vehicle motorcar
Precision: 1.0
Recall: 1.0
F1: 1.0
AP: 1.0

Query: computer
Expanded Query: computer computing device computing machine data processor electronic computer information processing system machine
Precision: 1.0
Recall: 1.0
F1: 1.0
AP: 1.0

Query: programming
Expanded Query: planning programing programming scheduling
Precision: 1.0
Recall: 0.5
F1: 0.6667
AP: 1.0


CELL 31 — Calculate MAP

In [ ]:
average_precisions = [result["AP"] for result in results]

map_score = sum(average_precisions) / len(average_precisions)

print("Average Precision Values:", average_precisions)
print("MAP:", round(map_score, 4))

Average Precision Values: [1.0, 1.0, 1.0]
MAP: 1.0


CELL 32 — Make a clean results table

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)

print("\nFinal Evaluation Results:\n")

display(
    results_df.round(4)
)

print("\nMean Average Precision (MAP):", round(map_score, 4))


Final Evaluation Results:



,Query,Precision,Recall,F1,AP
0,car,1.0,1.0,1.0000,1.0
1,computer,1.0,1.0,1.0000,1.0
2,programming,1.0,0.5,0.6667,1.0



Mean Average Precision (MAP): 1.0


CELL 33 — Keyword Search

In [ ]:
def keyword_search(query):

    query = query.lower()

    retrieved_documents = []

    for doc_id, document in zip(doc_ids, documents):

        if query in document.lower():
            retrieved_documents.append(doc_id)

    return retrieved_documents

CELL 34 — Test keyword search

In [ ]:
keyword_results = keyword_search("car")

print("Keyword Search Results:")
print(keyword_results)

Keyword Search Results:
['D2', 'D5']


CELL 35 — Correct exact keyword search

In [ ]:
import re

def keyword_search(query):

    query = query.lower()

    retrieved_documents = []

    for doc_id, document in zip(doc_ids, documents):

        words = re.findall(
            r'\b\w+\b',
            document.lower()
        )

        if query in words:
            retrieved_documents.append(doc_id)

    return retrieved_documents

In [ ]:
keyword_results = keyword_search("car")

print("Exact Keyword Search Results:")
print(keyword_results)

Exact Keyword Search Results:
['D2']


CELL 36 — Evaluate Keyword Search

In [ ]:
def evaluate_keyword_search(query, relevant_documents):

    retrieved_documents = keyword_search(query)

    retrieved_set = set(retrieved_documents)
    relevant_set = set(relevant_documents)

    true_positives = len(
        retrieved_set & relevant_set
    )

    precision = (
        true_positives / len(retrieved_set)
        if len(retrieved_set) > 0 else 0
    )

    recall = (
        true_positives / len(relevant_set)
        if len(relevant_set) > 0 else 0
    )

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = (
            2 * precision * recall
        ) / (precision + recall)

    return precision, recall, f1

CELL 37 — Evaluate the car query

In [ ]:
relevant_car_documents = queries["car"]

keyword_precision, keyword_recall, keyword_f1 = evaluate_keyword_search(
    "car",
    relevant_car_documents
)

print("Keyword Search Evaluation — car")
print("--------------------------------")

print(
    "Retrieved Documents:",
    keyword_search("car")
)

print(
    "Precision:",
    round(keyword_precision, 4)
)

print(
    "Recall:",
    round(keyword_recall, 4)
)

print(
    "F1 Score:",
    round(keyword_f1, 4)
)

Keyword Search Evaluation — car
--------------------------------
Retrieved Documents: ['D2']
Precision: 1.0
Recall: 0.2
F1 Score: 0.3333


CELL 38 — Create comparison table

In [ ]:
semantic_car_results = results_df[
    results_df["Query"] == "car"
].iloc[0]

comparison = pd.DataFrame({
    "Metric": [
        "Precision",
        "Recall",
        "F1 Score"
    ],

    "Keyword Search": [
        keyword_precision,
        keyword_recall,
        keyword_f1
    ],

    "WordNet + TF-IDF": [
        semantic_car_results["Precision"],
        semantic_car_results["Recall"],
        semantic_car_results["F1"]
    ]
})

display(comparison.round(4))

,Metric,Keyword Search,WordNet + TF-IDF
0,Precision,1.0000,1.0
1,Recall,0.2000,1.0
2,F1 Score,0.3333,1.0


Final comparison output

In [ ]:
# Final evaluation for the car query

query = "car"

ranked_documents, car_similarities, car_expanded_query = retrieve_documents(query)

car_precision, car_recall, car_f1 = calculate_metrics(
    ranked_documents,
    car_similarities,
    queries["car"]
)

car_ap = calculate_ap(
    ranked_documents,
    queries["car"]
)

print("=" * 60)
print("SEMANTIC INFORMATION RETRIEVAL USING WORDNET + TF-IDF")
print("=" * 60)

print("\nOriginal Query:", query)

print("\nExpanded Query:")
print(car_expanded_query)

print("\nExact Keyword Search:")
print("Retrieved:", keyword_search(query))
print("Precision:", round(keyword_precision, 4))
print("Recall:", round(keyword_recall, 4))
print("F1 Score:", round(keyword_f1, 4))

print("\nWordNet + TF-IDF:")
print("Ranked Results:")

for rank, doc_id in enumerate(ranked_documents, start=1):

    index = doc_ids.index(doc_id)

    print(
        f"{rank}. {doc_id} "
        f"(Score: {car_similarities[index]:.4f})"
    )

print("\nSemantic Search Evaluation:")
print("Precision:", round(car_precision, 4))
print("Recall:", round(car_recall, 4))
print("F1 Score:", round(car_f1, 4))
print("AP:", round(car_ap, 4))

print("\nOverall MAP:", round(map_score, 4))

print("=" * 60)

SEMANTIC INFORMATION RETRIEVAL USING WORDNET + TF-IDF

Original Query: car

Expanded Query:
auto automobile automotive vehicle car machine motor vehicle motorcar

Exact Keyword Search:
Retrieved: ['D2']
Precision: 1.0
Recall: 0.2
F1 Score: 0.3333

WordNet + TF-IDF:
Ranked Results:
1. D3 (Score: 0.3719)
2. D2 (Score: 0.2007)
3. D5 (Score: 0.1537)
4. D1 (Score: 0.1249)
5. D7 (Score: 0.1136)
6. D8 (Score: 0.0000)
7. D6 (Score: 0.0000)
8. D4 (Score: 0.0000)

Semantic Search Evaluation:
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
AP: 1.0

Overall MAP: 1.0
